In [1]:
# --- Imports ---
from config import CONFIG
from preprocessing import load_and_prepare
from training import run_training
from plotting import save_metrics, plot_confusion_matrix, plot_average_roc

print("Experiment settings:")
for k, v in CONFIG.items():
    if k != "split_files":
        print(f"{k}: {v}")
print(f"split_file: {CONFIG['split_files'][CONFIG['split_strategy']]}")

2025-10-09 16:42:42.346245: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-09 16:42:43.265112: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Experiment settings:
cell_line: HEPG2_new
aggregation: median
normalization: dmso_zscore
moa_condition: merge_st_vs_tk
split_strategy: structure
batch_size: 16
epochs: 1000
learning_rate: 0.0001
n_folds: 5
k_features: 300
data_root: ./Data_cp_features/
metadata_file: {cell_line}_metadata.xlsx
output_dir: ./results/
split_file: Splitting_Smiles/structure_merge_st_vs_tk_thr0.65_smiles.npz


In [2]:
# --- Preprocessing ---
print("Loading and preprocessing data...")
data_dict = load_and_prepare(CONFIG)
print("Preprocessing complete.")

Loading and preprocessing data...


/home/gauss/Documents/Project 2: Combining Molecular and Imagining Data for MOA Prediction/smile_image_global/Image-based Model/preprocessing.py:77: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df.loc[:, ~df.columns.str.contains(r"(_X$|_Y$|_Z$)")]
/home/gauss/Documents/Project 2: Combining Molecular and Imagining Data for MOA Prediction/smile_image_global/Image-based Model/preprocessing.py:77: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df.loc[:, ~df.columns.str.contains(r"(_X$|_Y$|_Z$)")]
/home/gauss/Documents/Project 2: Combining Molecular and Imagining Data for MOA Prediction/smile_image_global/Image-based Model/preprocessing.py:77: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df.loc[:, ~df.columns.str

Preprocessing complete.


In [3]:
def get_features(config, data_dict):
    trainval_df = data_dict["trainval_df"]
    test_df = data_dict["test_df"]
    feature_cols = data_dict["feature_cols"]

    # Extract feature arrays and labels
    X_trainval = trainval_df[feature_cols].to_numpy()
    y_trainval = trainval_df["moa"].to_numpy()
    X_test = test_df[feature_cols].to_numpy()
    y_test = test_df["moa"].to_numpy()

    # Encode labels
    all_labels = np.unique(np.concatenate([y_trainval, y_test]))
    label_to_int = {label: idx for idx, label in enumerate(all_labels)}
    int_to_label = {idx: label for label, idx in label_to_int.items()}
    y_trainval_enc = np.array([label_to_int[l] for l in y_trainval])
    y_test_enc = np.array([label_to_int[l] for l in y_test])

    # Stratified CV setup
    skf = StratifiedKFold(n_splits=config["n_folds"], shuffle=True, random_state=42)

    # store all folds for later inspection if you want
    all_fold_features = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_trainval, y_trainval_enc), 1):
        print(f"\n=== Fold {fold}/{config['n_folds']} ===")

        X_tr, X_val = X_trainval[train_idx], X_trainval[val_idx]
        y_tr, y_val = y_trainval_enc[train_idx], y_trainval_enc[val_idx]

        # --- Normalization ---
        if config["normalization"] == "standard":
            scaler = StandardScaler()
            X_tr = scaler.fit_transform(X_tr)
            X_val = scaler.transform(X_val)
            X_test_scaled = scaler.transform(X_test)
        else:
            X_tr, X_val, X_test_scaled = X_tr, X_val, X_test

        # --- Feature selection ---
        selector = SelectKBest(mutual_info_classif, k=config["k_features"])
        X_tr_sel = selector.fit_transform(X_tr, y_tr)
        X_val_sel = selector.transform(X_val)
        X_test_sel = selector.transform(X_test_scaled)

        # --- Selected features & scores ---
        selected_features = np.array(feature_cols)[selector.get_support()]
        selected_scores = selector.scores_[selector.get_support()]

        print(f"\nTop {len(selected_features)} selected features for Fold {fold}:")
        print(selected_features[:10])  # print only first 10 for brevity

        # --- Save per-fold feature list ---
        os.makedirs("fold_features", exist_ok=True)

        # Build a descriptive filename prefix using config
        prefix = (
            f"{config['cell_line']}_{config['aggregation']}_"
            f"{config['normalization']}_{config['split_strategy']}_"
            f"{config['moa_condition']}")
        
        # Save names
        np.savetxt(
            f"fold_features/{prefix}_fold{fold}_features.txt",
            selected_features,
            fmt="%s" )


        # keep in memory too
        all_fold_features.append(selected_features)

        # Save all folds combined for convenience
        np.save(f"fold_features/{prefix}_all_fold_features.npy", all_fold_features)

    print("\n✅ Saved selected feature lists for all folds in folder: 'fold_features/'")

    return all_fold_features


In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import SelectKBest, mutual_info_classif
import os
from sklearn.preprocessing import StandardScaler


get_features(CONFIG, data_dict)


=== Fold 1/5 ===

Top 300 selected features for Fold 1:
['Granularity_10_illumMITO' 'Granularity_11_illumCONC'
 'Granularity_11_illumMITO' 'Granularity_12_illumCONC'
 'Granularity_13_illumCONC' 'Granularity_2_illumHOECHST'
 'Granularity_4_illumHOECHST' 'Mean_cells_AreaShape_Area'
 'Mean_cells_AreaShape_BoundingBoxArea' 'Mean_cells_AreaShape_ConvexArea']

=== Fold 2/5 ===

Top 300 selected features for Fold 2:
['Granularity_10_illumCONC' 'Granularity_10_illumMITO'
 'Granularity_11_illumCONC' 'Granularity_2_illumHOECHST'
 'Granularity_2_illumSYTO' 'Granularity_3_illumHOECHST'
 'Granularity_4_illumHOECHST' 'Granularity_9_illumCONC'
 'Granularity_9_illumHOECHST' 'Mean_cells_AreaShape_Area']

=== Fold 3/5 ===

Top 300 selected features for Fold 3:
['Granularity_1_illumCONC' 'Granularity_2_illumHOECHST'
 'Mean_cells_AreaShape_Area' 'Mean_cells_AreaShape_BoundingBoxArea'
 'Mean_cells_AreaShape_ConvexArea' 'Mean_cells_AreaShape_Eccentricity'
 'Mean_cells_AreaShape_EquivalentDiameter'
 'Mean_c

[array(['Granularity_10_illumMITO', 'Granularity_11_illumCONC',
        'Granularity_11_illumMITO', 'Granularity_12_illumCONC',
        'Granularity_13_illumCONC', 'Granularity_2_illumHOECHST',
        'Granularity_4_illumHOECHST', 'Mean_cells_AreaShape_Area',
        'Mean_cells_AreaShape_BoundingBoxArea',
        'Mean_cells_AreaShape_ConvexArea',
        'Mean_cells_AreaShape_Eccentricity',
        'Mean_cells_AreaShape_EquivalentDiameter',
        'Mean_cells_AreaShape_MajorAxisLength',
        'Mean_cells_AreaShape_MaxFeretDiameter',
        'Mean_cells_AreaShape_MaximumRadius',
        'Mean_cells_AreaShape_MeanRadius',
        'Mean_cells_AreaShape_MedianRadius',
        'Mean_cells_AreaShape_MinFeretDiameter',
        'Mean_cells_AreaShape_MinorAxisLength',
        'Mean_cells_AreaShape_Perimeter',
        'Mean_cells_AreaShape_Zernike_2_0',
        'Mean_cells_AreaShape_Zernike_4_0',
        'Mean_cells_AreaShape_Zernike_5_5',
        'Mean_cells_Correlation_Correlation_illumC